In [1]:
# Import libraries 
import pandas as pd 
import numpy as np
import os

In [2]:
# Load dataset 
df = pd.read_csv("../data/raw/disease_dataset.csv") 
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1500, 15)


,age,gender,temperature,heart_rate,blood_pressure,glucose,cholesterol,fever,cough,fatigue,headache,chest_pain,breathing_difficulty,nausea,disease
0,65.0,Female,98.3,71,176.0,130.0,236,Yes,No,No,NaN,Yes,Yes,Yes,Pneumonia
1,19.0,Female,97.6,72,124.0,115.0,132,No,No,No,Yes,Yes,Yes,No,Migraine
2,NaN,Female,98.9,96,145.0,154.0,244,Yes,Yes,No,No,No,Yes,Yes,Hypertension
3,45.0,Male,99.5,99,168.0,172.0,275,Yes,Yes,Yes,No,No,No,Yes,Diabetes
4,70.0,Male,100.6,71,142.0,NaN,181,Yes,Yes,No,No,Yes,No,Yes,Flu


In [3]:
# Check columns 
print("Columns:")
print(df.columns.tolist())  

Columns:
['age', 'gender', 'temperature', 'heart_rate', 'blood_pressure', 'glucose', 'cholesterol', 'fever', 'cough', 'fatigue', 'headache', 'chest_pain', 'breathing_difficulty', 'nausea', 'disease']


In [4]:
# Define feature engineering function 
def create_features(df):
    df = df.copy()
    symptom_columns = [
        "fever",
        "cough",
        "fatigue",
        "headache",
        "chest_pain",
        "breathing_difficulty",
        "nausea"
    ]
    # Convert symptoms temporarily to 0/1
    binary_symptoms = df[symptom_columns].apply(
        lambda col: col.map({
            "Yes": 1,
            "No": 0,
            1: 1,
            0: 0
        })
    )
    # Number of symptoms
    df["symptom_count"] = binary_symptoms.sum(axis=1)
    # Age group
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 18, 40, 60, 100],
        labels=[
            "Child",
            "Young_Adult",
            "Middle_Aged",
            "Senior"
        ],
        include_lowest=True
    )
    # Blood pressure category
    df["blood_pressure_category"] = pd.cut(
        df["blood_pressure"],
        bins=[0, 90, 120, 140, 200],
        labels=[
            "Low",
            "Normal",
            "Elevated",
            "High"
        ],
        include_lowest=True
    )
    # Glucose category
    df["glucose_category"] = pd.cut(
        df["glucose"],
        bins=[0, 70, 100, 126, 300],
        labels=[
            "Low",
            "Normal",
            "Prediabetic",
            "High"
        ],
        include_lowest=True
    ) 
    return df 


In [5]:
# Apply feature engineering 
featured_df = create_features(df)
print("Original Shape:", df.shape)
print("Feature Engineered Shape:", featured_df.shape) 
featured_df.head() 

Original Shape: (1500, 15)
Feature Engineered Shape: (1500, 19)


,age,gender,temperature,heart_rate,blood_pressure,glucose,cholesterol,fever,cough,fatigue,headache,chest_pain,breathing_difficulty,nausea,disease,symptom_count,age_group,blood_pressure_category,glucose_category
0,65.0,Female,98.3,71,176.0,130.0,236,Yes,No,No,NaN,Yes,Yes,Yes,Pneumonia,4.0,Senior,High,High
1,19.0,Female,97.6,72,124.0,115.0,132,No,No,No,Yes,Yes,Yes,No,Migraine,3.0,Young_Adult,Elevated,Prediabetic
2,NaN,Female,98.9,96,145.0,154.0,244,Yes,Yes,No,No,No,Yes,Yes,Hypertension,4.0,NaN,High,High
3,45.0,Male,99.5,99,168.0,172.0,275,Yes,Yes,Yes,No,No,No,Yes,Diabetes,4.0,Middle_Aged,High,High
4,70.0,Male,100.6,71,142.0,NaN,181,Yes,Yes,No,No,Yes,No,Yes,Flu,4.0,Senior,High,NaN


In [6]:
# Check new columns 
print("Columns after feature engineering:") 
for column in featured_df.columns:
    print(column) 

Columns after feature engineering:
age
gender
temperature
heart_rate
blood_pressure
glucose
cholesterol
fever
cough
fatigue
headache
chest_pain
breathing_difficulty
nausea
disease
symptom_count
age_group
blood_pressure_category
glucose_category


In [7]:
# Check engineered features 
print("Symptom Count:")
display(featured_df["symptom_count"].value_counts().sort_index()) 
print("\nAge Groups:")
display(featured_df["age_group"].value_counts()) 
print("\nBlood Pressure Categories:")
display(featured_df["blood_pressure_category"].value_counts()) 
print("\nGlucose Categories:")
display(featured_df["glucose_category"].value_counts()) 

Symptom Count:


symptom_count
0.0      3
1.0     59
2.0    181
3.0    389
4.0    440
5.0    306
6.0    105
7.0     17
Name: count, dtype: int64


Age Groups:


age_group
Young_Adult    542
Middle_Aged    450
Senior         447
Child           16
Name: count, dtype: int64


Blood Pressure Categories:


blood_pressure_category
High        816
Normal      374
Elevated    248
Low          17
Name: count, dtype: int64


Glucose Categories:


glucose_category
High           1013
Normal          238
Prediabetic     193
Low              11
Name: count, dtype: int64

In [8]:
# Check missing values 
print("Missing values after feature engineering:") 
print(
    featured_df[
        [
            "symptom_count",
            "age_group",
            "blood_pressure_category",
            "glucose_category"
        ]
    ].isnull().sum()
)  

Missing values after feature engineering:
symptom_count               0
age_group                  45
blood_pressure_category    45
glucose_category           45
dtype: int64


In [9]:
# Save processed dataset 
os.makedirs("../data/processed", exist_ok=True)
featured_df.to_csv(
    "../data/processed/processed_data.csv",
    index=False
) 
print("Processed dataset saved successfully.")
print("../data/processed/processed_data.csv") 

Processed dataset saved successfully.
../data/processed/processed_data.csv


In [10]:
# Final verification 
processed_df = pd.read_csv(
    "../data/processed/processed_data.csv"
)
print("Processed Dataset Shape:", processed_df.shape)
print("\nProcessed Dataset Columns:")
print(processed_df.columns.tolist()) 
processed_df.head() 

Processed Dataset Shape: (1500, 19)

Processed Dataset Columns:
['age', 'gender', 'temperature', 'heart_rate', 'blood_pressure', 'glucose', 'cholesterol', 'fever', 'cough', 'fatigue', 'headache', 'chest_pain', 'breathing_difficulty', 'nausea', 'disease', 'symptom_count', 'age_group', 'blood_pressure_category', 'glucose_category']


,age,gender,temperature,heart_rate,blood_pressure,glucose,cholesterol,fever,cough,fatigue,headache,chest_pain,breathing_difficulty,nausea,disease,symptom_count,age_group,blood_pressure_category,glucose_category
0,65.0,Female,98.3,71,176.0,130.0,236,Yes,No,No,NaN,Yes,Yes,Yes,Pneumonia,4.0,Senior,High,High
1,19.0,Female,97.6,72,124.0,115.0,132,No,No,No,Yes,Yes,Yes,No,Migraine,3.0,Young_Adult,Elevated,Prediabetic
2,NaN,Female,98.9,96,145.0,154.0,244,Yes,Yes,No,No,No,Yes,Yes,Hypertension,4.0,NaN,High,High
3,45.0,Male,99.5,99,168.0,172.0,275,Yes,Yes,Yes,No,No,No,Yes,Diabetes,4.0,Middle_Aged,High,High
4,70.0,Male,100.6,71,142.0,NaN,181,Yes,Yes,No,No,Yes,No,Yes,Flu,4.0,Senior,High,NaN
